# Face Mask Detection — Colab Training

Runs all three experiments on a Colab GPU and exports `results/` back to Google Drive so you can download it to your laptop.

## Before running
1. **Runtime → Change runtime type → T4 GPU**
2. In your Google Drive, create a folder `FaceMask/` and upload:
   - The project **code** — either a zip of `src/` + `run_all.py` + `requirements.txt`, or clone from GitHub in the cell below.
   - The **dataset** as a zip: `dataset_processed.zip` (containing `Train/`, `Validation/`, `Test/`).

Expected Drive layout:
```
MyDrive/FaceMask/
├── project.zip          # contains src/, run_all.py
└── dataset_processed.zip
```

## 1. Verify GPU

In [ ]:
!nvidia-smi
import tensorflow as tf
print('TF', tf.__version__, 'GPUs:', tf.config.list_physical_devices('GPU'))

Tue Apr 14 05:12:21 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 3. Load code + dataset

Pick **ONE** of the two approaches below and run only that cell.

### Option A: From zip in Drive (recommended)

In [ ]:
import os, shutil, subprocess

DRIVE_DIR = '/content/drive/MyDrive'
WORK      = '/content/project'
DATA      = '/content/dataset_processed'

# Clean slate
if os.path.exists(WORK):
    shutil.rmtree(WORK)
os.makedirs(WORK)

# Unzip project code
subprocess.run(['unzip', '-q', '-o', f'{DRIVE_DIR}/project.zip', '-d', WORK], check=True)

# Unzip dataset
subprocess.run(['unzip', '-q', '-o', f'{DRIVE_DIR}/dataset_processed.zip', '-d', '/content'], check=True)

# Flatten: if zip had a single top-level folder, move contents up
entries = [e for e in os.listdir(WORK) if not e.startswith('.')]
if len(entries) == 1 and os.path.isdir(os.path.join(WORK, entries[0])):
    inner = os.path.join(WORK, entries[0])
    for item in os.listdir(inner):
        shutil.move(os.path.join(inner, item), os.path.join(WORK, item))
    shutil.rmtree(inner)
    print(f'Flattened: {entries[0]}/ -> {WORK}/')

print('\n--- Project files ---')
for f in sorted(os.listdir(WORK)):
    print(f'  {f}{"/ " if os.path.isdir(os.path.join(WORK,f)) else ""}')

print(f'\n--- Dataset dirs ---')
if os.path.exists(DATA):
    print([d for d in os.listdir(DATA) if os.path.isdir(os.path.join(DATA,d))])
else:
    print(f'WARNING: {DATA} not found! Check your zip structure.')

# Verify run_all.py exists
assert os.path.exists(os.path.join(WORK, 'run_all.py')), \
    f'run_all.py not found in {WORK}! Contents: {os.listdir(WORK)}'

Flattened: Project/ -> /content/project/

--- Project files ---
  requirements.txt
  run_all.py
  src/ 

--- Dataset dirs ---
['Validation', 'Train', 'Test']


### Option B: From GitHub repo

In [ ]:
# Replace <YOUR_USER>/<YOUR_REPO> and unzip the dataset from Drive.
# WORK = '/content/project'
# DATA = '/content/dataset_processed'
# !git clone https://github.com/<YOUR_USER>/<YOUR_REPO>.git $WORK
# !unzip -q -o '/content/drive/MyDrive/FaceMask/dataset_processed.zip' -d /content

## 4. Install extra deps (most are pre-installed on Colab)

In [ ]:
!pip install -q opencv-python seaborn scikit-learn
# tensorflow_hub is needed only for the TF-Hub ViT; skip if you pass --skip_vit
!pip install -q tensorflow-hub

## 5. Train experiments

Each experiment runs in its own cell so you can execute them independently.

`--fast` enables mixed_float16 + XLA on the Colab GPU.

In [ ]:
%cd /content/project
!python run_all.py --only 1 --data_dir /content/dataset_processed --fast

/content/project

################################################################
#  EXPERIMENT 1 - Custom CNN Architectures (Supervised Learning)#
################################################################

2026-04-14 05:21:49.729866: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776144109.763554   13548 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776144109.774542   13548 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776144109.798912   13548 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776144109.798947   13548 computati

### Experiment 1 — Custom CNN
Expect ~10–15 min on a T4.

In [ ]:
%cd /content/project
!python run_all.py --only 2 --data_dir /content/dataset_processed --fast

/content/project

################################################################
#  EXPERIMENT 2 - Transfer Learning (Unsupervised Feature Extraction)#
################################################################

2026-04-14 05:29:03.070509: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776144543.092620   18397 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776144543.101642   18397 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776144543.120378   18397 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776144543.120422   18397 comp

### Experiment 2 — Transfer Learning
Expect ~10–20 min on a T4.

### Experiment 3 — EfficientNetB0
Expect ~10–20 min on a T4. Add `--skip_vit` if you don't need the ViT comparison.

In [ ]:
%cd /content/project
!python run_all.py --only 3 --data_dir /content/dataset_processed --fast

/content/project

################################################################
#  EXPERIMENT 3 - State-of-the-Art Models (EfficientNet + ViT) #
################################################################

2026-04-14 05:39:26.214251: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776145166.235805   22795 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776145166.242900   22795 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776145166.259563   22795 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776145166.259613   22795 computatio

## 6. Export results back to Drive

This copies everything in `results/` to `MyDrive/FaceMask/results_<timestamp>/` so you can download it to your laptop from Drive.

In [ ]:
import os, shutil, datetime
stamp = datetime.datetime.now().strftime('%Y%m%d_%H%M')
src   = '/content/project/results'
dst   = f'/content/drive/MyDrive/FaceMask/results_{stamp}'
shutil.copytree(src, dst)
print('Exported ->', dst)

# Also create a single zip for easy download
zip_path = f'/content/drive/MyDrive/FaceMask/results_{stamp}.zip'
shutil.make_archive(zip_path.replace('.zip', ''), 'zip', src)
print('Zipped   ->', zip_path)

Exported -> /content/drive/MyDrive/FaceMask/results_20260414_0557
Zipped   -> /content/drive/MyDrive/FaceMask/results_20260414_0557.zip


## 7. (Optional) download results directly to this browser

In [ ]:
from google.colab import files
files.download(zip_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>